In [ ]:
import pandas as pd
import numpy as np
import gc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import OneHotEncoder
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import koreanize_matplotlib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from autogluon.tabular import TabularPredictor, TabularDataset
import random
import os

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(36) 

In [ ]:
# 데이터 분할(폴더) 구분
data_splits = ["train", "test"]

# 각 데이터 유형별 폴더명, 파일 접미사, 변수 접두어 설정
data_categories = {
    "회원정보": {"suffix": "회원정보", "var_prefix": "customer"},
    "신용정보": {"suffix": "신용정보", "var_prefix": "credit"},
    "승인매출정보": {"suffix": "승인매출정보", "var_prefix": "sales"},
    "청구정보": {"suffix": "청구정보", "var_prefix": "billing"},
    "잔액정보": {"suffix": "잔액정보", "var_prefix": "balance"},
    "채널정보": {"suffix": "채널정보", "var_prefix": "channel"},
    "마케팅정보": {"suffix": "마케팅정보", "var_prefix": "marketing"},
    "성과정보": {"suffix": "성과정보", "var_prefix": "performance"}
}

In [ ]:
# 2018년 7월부터 12월까지의 월 리스트
months = ['07', '08', '09', '10', '11', '12']

for split in data_splits:
    for category, info in data_categories.items():
        suffix = info["suffix"]
        var_prefix = info["var_prefix"]

        for month in months:
            # 파일명 형식: 2018{month}_{split}_{suffix}.parquet
            file_path = f"./2018{month}_{split}_{suffix}.parquet"
            # 변수명 형식: {var_prefix}_{split}_{month}
            variable_name = f"{var_prefix}_{split}_{month}"
            globals()[variable_name] = pd.read_parquet(file_path)
            print(f"{variable_name} is loaded from {file_path}")

gc.collect()

In [ ]:
# 데이터 유형별 설정
info_categories = ["customer", "credit", "sales", "billing", "balance", "channel", "marketing", "performance"]

# 월 설정
months = ['07', '08', '09', '10', '11', '12']

#### Train ####

# 각 유형별로 월별 데이터를 합쳐서 새로운 변수에 저장
train_dfs = {}

for prefix in info_categories:
    # globals()에서 동적 변수명으로 데이터프레임들을 가져와 리스트에 저장
    df_list = [globals()[f"{prefix}_train_{month}"] for month in months]
    train_dfs[f"{prefix}_train_df"] = pd.concat(df_list, axis=0)
    gc.collect()
    print(f"{prefix}_train_df is created with shape: {train_dfs[f'{prefix}_train_df'].shape}")


customer_train_df = train_dfs["customer_train_df"]
credit_train_df   = train_dfs["credit_train_df"]
sales_train_df    = train_dfs["sales_train_df"]
billing_train_df  = train_dfs["billing_train_df"]
balance_train_df  = train_dfs["balance_train_df"]
channel_train_df  = train_dfs["channel_train_df"]
marketing_train_df= train_dfs["marketing_train_df"]
performance_train_df = train_dfs["performance_train_df"]

gc.collect()

In [ ]:

#### Test ####

# test 데이터에 대해 train과 동일한 방법 적용
test_dfs = {}

for prefix in info_categories:
    df_list = [globals()[f"{prefix}_test_{month}"] for month in months]
    test_dfs[f"{prefix}_test_df"] = pd.concat(df_list, axis=0)
    gc.collect()
    print(f"{prefix}_test_df is created with shape: {test_dfs[f'{prefix}_test_df'].shape}")


customer_test_df = test_dfs["customer_test_df"]
credit_test_df   = test_dfs["credit_test_df"]
sales_test_df    = test_dfs["sales_test_df"]
billing_test_df  = test_dfs["billing_test_df"]
balance_test_df  = test_dfs["balance_test_df"]
channel_test_df  = test_dfs["channel_test_df"]
marketing_test_df= test_dfs["marketing_test_df"]
performance_test_df = test_dfs["performance_test_df"]

gc.collect()

In [ ]:
#### Train ####

train_df = customer_train_df.merge(credit_train_df, on=['기준년월', 'ID'], how='left')
print("Step1 저장 완료: train_step1, shape:", train_df.shape)
del customer_train_df, credit_train_df
gc.collect()

# 이후 merge할 데이터프레임 이름과 단계 정보를 리스트에 저장
merge_list = [
    ("sales_train_df",    "Step2"),
    ("billing_train_df",  "Step3"),
    ("balance_train_df",  "Step4"),
    ("channel_train_df",  "Step5"),
    ("marketing_train_df","Step6"),
    ("performance_train_df", "최종")
]

# 나머지 단계 merge
for df_name, step in tqdm(merge_list, desc="Merging DataFrames"):
    # globals()로 동적 변수 접근하여 merge 수행
    train_df = train_df.merge(globals()[df_name], on=['기준년월', 'ID'], how='left')
    print(f"{step} 저장 완료: train_{step}, shape:", train_df.shape)
    # 사용한 변수는 메모리 해제를 위해 삭제
    del globals()[df_name]
    gc.collect()

In [ ]:
merge_list

In [ ]:
#### Test ####

test_df = customer_test_df.merge(credit_test_df, on=['기준년월', 'ID'], how='left')
print("Step1 저장 완료: test_step1, shape:", test_df.shape)
del customer_test_df, credit_test_df
gc.collect()

# 이후 merge할 데이터프레임 이름과 단계 정보를 리스트에 저장
merge_list = [
    ("sales_test_df",    "Step2"),
    ("billing_test_df",  "Step3"),
    ("balance_test_df",  "Step4"),
    ("channel_test_df",  "Step5"),
    ("marketing_test_df","Step6"),
    ("performance_test_df", "최종")
]

for df_name, step in tqdm(merge_list, desc="Merging DataFrames"):
    test_df = test_df.merge(globals()[df_name], on=['기준년월', 'ID'], how='left')
    print(f"{step} 저장 완료: test_{step}, shape:", test_df.shape)
    del globals()[df_name]
    gc.collect()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 예시 데이터프레임 생성 (실제 데이터프레임을 사용하세요)
# train_df = pd.read_csv('your_data.csv')

# Segment의 고유 값과 그 수를 계산
segment_counts = train_df["Segment"].value_counts()
segment_percentages = segment_counts / segment_counts.sum() * 100

# 바 그래프 시각화
plt.figure(figsize=(10, 6))
bars = plt.bar(segment_counts.index, segment_counts.values, color='skyblue')

# 각 바 위에 비율 표시
for bar in bars:
    yval = bar.get_height()
    percentage = (yval / segment_counts.sum()) * 100
    plt.text(bar.get_x() + bar.get_width() / 2, yval, f'{percentage:.4f}%', ha='center', va='bottom')

plt.xlabel('Segment')
plt.ylabel('Count')
plt.title('Count of Segments')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# # 타겟 변수 설정
target = "Segment"

# ID 컬럼과 음수 중요도 컬럼 제거
train = TabularDataset(train_df.drop(columns=['ID']))
test = TabularDataset(test_df.drop(columns=['ID']))

In [ ]:
from autogluon.tabular import TabularPredictor

operation_time = 12 * 60 * 60  # 12시간
exclude_model_types = [
    'KNN',  # K-Nearest Neighbors
    'RF',   # Random Forest
    'XT',   # Extra Trees
    'LR',   # Linear Regression
    'NN'    # Tabular Neural Network
]

predictor = TabularPredictor(label=target, eval_metric='f1_macro').fit(
    train_data=train,
    presets='best_quality', 
    ag_args_fit={
        "num_cpus": 32,      # CPU도 병렬로 활용
        "num_gpus": 1,       # Titan RTX 한 장 사용
    },
    num_bag_folds=5,         # 앙상블 성능 향상
    num_stack_levels = 2,      # 딥 모델 포함해서 스태킹 한 단계 수행
    time_limit=operation_time,
    verbosity=3,
    excluded_model_types=exclude_model_types
)


In [ ]:
# predictor = TabularPredictor.load("./AutogluonModels/ag-20250416_222000")
ld_board = predictor.leaderboard(train, silent=True)

In [ ]:
ld_board

In [ ]:
train_transformed = predictor.transform_features(train)
train_transformed.head()

In [ ]:
# feature importance 계산 (default는 best 모델 기준)
importance_df = predictor.feature_importance(train, model=model_to_use)

# 중요도 높은 순으로 상위 10개 출력
importance_df.head(10)

In [ ]:
# importance가 음수인 feature만 필터링
negative_importance_df = importance_df[importance_df['importance'] < 0]

In [ ]:
# 타겟 변수 설정
target = "Segment"

# 음수 importance feature 불러오기
neg_feat_df = pd.read_csv("negative_importance_features.csv", index_col=0)
neg_feat_cols = neg_feat_df.index.tolist()  # feature 이름 리스트로 추출

# ID 컬럼과 음수 중요도 컬럼 제거
train = TabularDataset(train_df.drop(columns=['ID'] + neg_feat_cols))
test = TabularDataset(test_df.drop(columns=['ID'] + neg_feat_cols))

print(f"제거된 feature 수: {len(neg_feat_cols)}개")

In [ ]:
from autogluon.tabular import TabularPredictor

operation_time = 12 * 60 * 60  # 12시간
exclude_model_types = [
    'KNN',  # K-Nearest Neighbors
    'RF',   # Random Forest
    'XT',   # Extra Trees
    'LR',   # Linear Regression
    'NN'    # Tabular Neural Network
]

predictor = TabularPredictor(label=target, eval_metric='f1_macro').fit(
    train_data=train,
    presets='best_quality', 
    ag_args_fit={
        "num_cpus": 32,      # CPU도 병렬로 활용
        "num_gpus": 1,       # Titan RTX 한 장 사용
    },
    num_bag_folds=5,         # 앙상블 성능 향상
    num_stack_levels = 2,      # 딥 모델 포함해서 스태킹 한 단계 수행
    time_limit=operation_time,
    verbosity=3,
    excluded_model_types=exclude_model_types
)


In [ ]:
ld_board = predictor.leaderboard(train, silent=True)

In [ ]:
model_to_use = predictor.model_best
y_test_pred = predictor.predict(test, model=model_to_use)

In [ ]:
len(y_test_pred)

In [ ]:
test.shape

In [ ]:
test_df.shape

In [ ]:
# row 단위 예측 결과를 test_data에 추가
test_data = test_df.copy()  # 원본 유지
test_data["pred_label"] = y_test_pred

In [ ]:
test_data["pred_label"]

In [ ]:
submission = test_data.groupby("ID")["pred_label"] \
    .agg(lambda x: x.value_counts().idxmax()) \
    .reset_index()

submission.columns = ["ID", "Segment"]

submission.to_csv('./Segment_autogluon_final.csv', index=False)